<a href="https://colab.research.google.com/github/hemanthrajelangovan07-sudo/smoltalk-rlhf-pipeline/blob/main/2_SFT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth
from unsloth import FastLanguageModel
import torch

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048
LORA_RANK = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

print(f"✅ Base model loaded")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"   Dtype: {next(model.parameters()).dtype}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base model loaded
   Parameters: 1018.0M
   Dtype: torch.float16


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,
    loftq_config=None,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ LoRA adapters attached")
print(f"   Total params:     {total_params / 1e6:.2f}M")
print(f"   Trainable params: {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.7 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



✅ LoRA adapters attached
   Total params:     1091.84M
   Trainable params: 73.86M (6.76%)


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

sft_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-rewrite", split="train[:15000]")

def apply_chat_template(examples):
    """
    Convert messages list → formatted string with special tokens.
    """
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

sft_formatted = sft_dataset.map(
    apply_chat_template,
    batched=True,
    batch_size=1000,
    remove_columns=sft_dataset.column_names,
    desc="Applying chat template",
)

print(f"\n✅ Chat template applied")
print(f"Sample:\n{sft_formatted[0]['text'][:400]}")

README.md: 0.00B [00:00, ?B/s]

data/smol-rewrite/train-00000-of-00001.p(…):   0%|          | 0.00/38.9M [00:00<?, ?B/s]

data/smol-rewrite/test-00000-of-00001.pa(…):   0%|          | 0.00/2.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/53342 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2808 [00:00<?, ? examples/s]

Applying chat template:   0%|          | 0/15000 [00:00<?, ? examples/s]


✅ Chat template applied
Sample:
<|im_start|>system
You're an AI assistant for text re-writing. Rewrite the input text to make it more concise while preserving its core meaning.<|im_end|>
<|im_start|>user
Hey Alex,

I hope you're doing well! It's been a while since we met at the film festival last year. I was the one with the short film about the old abandoned factory. Anyway, I'm reaching out because I'm currently working on my 


In [ ]:
import os
import wandb

BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"

drive_out = f"{BASE_DIR}/checkpoints/sft"
os.makedirs(drive_out, exist_ok=True)
with open(f"{drive_out}/.write_test", "w") as f: f.write("ok")
os.remove(f"{drive_out}/.write_test")
print("✅ Drive writable — safe to start training")

wandb.init(
    project="rlhf-simpo-grpo-2026",
    name="01_sft_qwen25-1.5b_smoltalk15k",
    config={
        "model": MODEL_ID,
        "stage": "SFT",
        "dataset": "smoltalk_15k",
        "lora_rank": LORA_RANK,
        "max_seq_len": MAX_SEQ_LEN,
    }
)

sft_config = SFTConfig(

    output_dir=drive_out,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,


    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,


    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    max_grad_norm=1.0,
    gradient_checkpointing=True,


    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=True,
    dataset_num_proc=2,


    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="wandb",


    eval_strategy="no",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=sft_formatted,
)

print("✅ Trainer ready")
print(f"   Output dir : {drive_out}")
print(f"   Checkpoints: every 100 steps → Drive")
print(f"   Steps total: ~306 | Epochs: 2 | EBS: 16")

✅ Drive writable — safe to start training


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hemanthrajelangovan07 (hemanthrajelangovan07-sathyabama-institute-of-science-an) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✅ Trainer ready
   Output dir : /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft
   Checkpoints: every 100 steps → Drive
   Steps total: ~306 | Epochs: 2 | EBS: 16


In [ ]:
import time, os

print("🚀 Starting SFT training...")
print(f"   Steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")
print(f"   Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"   Estimated time: ~45 min on T4")

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"\n✅ SFT Training complete in {elapsed/60:.1f} minutes")
print(f"   Final loss: {train_result.training_loss:.4f}")

final_out = f"{BASE_DIR}/checkpoints/sft/final_adapter"
os.makedirs(final_out, exist_ok=True)

model.save_pretrained(final_out)
tokenizer.save_pretrained(final_out)

required = {"adapter_config.json", "adapter_model.safetensors"}
saved = set(os.listdir(final_out))
missing = required - saved
if missing:
    raise RuntimeError(f"❌ Save incomplete — missing files: {missing}\nSaved: {saved}")

print(f"✅ SFT adapter verified on Drive: {saved}")

ckpt_dir = f"{BASE_DIR}/checkpoints/sft"
checkpoints = [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")]
print(f"✅ Checkpoints on Drive: {sorted(checkpoints)}")

wandb.finish()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting SFT training...
   Steps: auto
   Effective batch size: 16
   Estimated time: ~45 min on T4


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,438 | Num Epochs = 2 | Total steps = 306
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 73,859,072 of 1,617,573,376 (4.57% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.259871
20,0.975486
30,0.920894
40,0.878644
50,0.870199
60,0.850577
70,0.850453
80,0.845505
90,0.826485
100,0.822432


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft/checkpoint-306/tokenizer_config.json.



✅ SFT Training complete in 112.2 minutes
   Final loss: 0.7163


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/unsloth_checkpoints/checkpoints/sft/final_adapter/tokenizer_config.json.


✅ SFT adapter verified on Drive: {'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json', 'adapter_model.safetensors'}
✅ Checkpoints on Drive: ['checkpoint-200', 'checkpoint-300', 'checkpoint-306']


train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/grad_norm,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▅▅▄▄▄▄▄▄▄▄▄▃▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,8.138971201468416e+16
train/epoch,2
train/global_step,306
train/grad_norm,0.88173
train/learning_rate,0.0
train/loss,0.54084


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"
sft_dir = f"{BASE_DIR}/checkpoints/sft"

print("=" * 50)
print("📁 SFT DRIVE VERIFICATION")
print("=" * 50)

final = f"{sft_dir}/final_adapter"
required = {"adapter_config.json", "adapter_model.safetensors", "tokenizer_config.json"}

if os.path.exists(final):
    files = set(os.listdir(final))
    missing = required - files
    if missing:
        print(f"⚠️  final_adapter exists but incomplete — missing: {missing}")
    else:
        sizes = {f: f"{os.path.getsize(f'{final}/{f}') / 1e6:.1f} MB" for f in files}
        print(f"✅ final_adapter — all files present:")
        for name, size in sorted(sizes.items()):
            print(f"   {name}: {size}")
else:
    print("❌ final_adapter NOT found on Drive")

print()

checkpoints = sorted([d for d in os.listdir(sft_dir) if d.startswith("checkpoint-")])
if checkpoints:
    print(f"✅ Mid-run checkpoints found: {checkpoints}")
    for ckpt in checkpoints:
        ckpt_path = f"{sft_dir}/{ckpt}"
        ckpt_files = os.listdir(ckpt_path)
        size_mb = sum(os.path.getsize(f"{ckpt_path}/{f}") for f in ckpt_files) / 1e6
        print(f"   {ckpt}: {len(ckpt_files)} files, {size_mb:.1f} MB total")
else:
    print("❌ No mid-run checkpoints found on Drive")

print()

import json
config_path = f"{final}/adapter_config.json"
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    print(f"✅ adapter_config.json readable:")
    print(f"   base model : {cfg.get('base_model_name_or_path')}")
    print(f"   peft type  : {cfg.get('peft_type')}")
    print(f"   lora rank  : {cfg.get('r')}")
    print(f"   lora alpha : {cfg.get('lora_alpha')}")

print()
print("=" * 50)
print("✅ Verification complete" if os.path.exists(final) else "❌ Verification failed")
print("=" * 50)

In [2]:
import nbformat

def clean_notebook_metadata(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Remove the widgets metadata which causes the rendering error
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
        print(f"✅ Removed 'widgets' from metadata.")
    else:
        print("ℹ️ No 'widgets' key found in metadata.")

    with open(output_file, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f"✅ Cleaned notebook saved to: {output_file}")

# Usage: Replace with your actual filename if you've uploaded it to Colab
# clean_notebook_metadata('your_notebook.ipynb', 'cleaned_notebook.ipynb')